<a href="https://colab.research.google.com/github/belinatom/NALAPROJECT/blob/main/nlp_task4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 4 — Llama-3 Zero-shot & Few-shot Classification

**Author**: Belinda Mziray | HSLU MSc IT, Digitalization & Sustainability | 2026


## Approach
- **Zero-shot**: Prompt contains all 56 category names with Swahili descriptions; model classifies without examples.

- **Few-shot**: Prompt includes 2 example proverbs for 10 categories; model classifies via in-context learning.







In [ ]:
# Install & imports
!pip install -q gdown wandb mlflow

import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, classification_report
from huggingface_hub import login
from getpass import getpass

import gdown

seed = 42
np.random.seed(seed); random.seed(seed); torch.manual_seed(seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sns.set_style('whitegrid')
print(f'✓ Imports loaded | Device: {device}')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.3/86.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 907.5/907.5 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.

In [ ]:
# Authenticate with HuggingFace (required for Llama-3)
hf_token = getpass('Paste your HuggingFace token (huggingface.co/settings/tokens): ')
login(token=hf_token)
print('✓ Authenticated with HuggingFace')


Paste your HuggingFace token (huggingface.co/settings/tokens): ··········
✓ Authenticated with HuggingFace


In [ ]:
# Download dataset from Google Drive
FILE_ID = '1Rp9Fg1BitlAR3d1JyH4DmTp3aSKr8H1t'
gdown.download(
    f'https://drive.google.com/uc?id={FILE_ID}',
    '/content/swahiliproverbs.csv',
    quiet=False, fuzzy=True
)

df = pd.read_csv('/content/swahiliproverbs.csv')
df = df[['swahili_proverb', 'label']].dropna().copy()
df['swahili_proverb'] = df['swahili_proverb'].astype(str).str.strip()
df['label']           = df['label'].astype(str).str.strip()
df = df[df['swahili_proverb'] != ''].reset_index(drop=True)

text_col, label_col = 'swahili_proverb', 'label'
le = LabelEncoder()
le.fit(df[label_col])
all_labels = sorted(df[label_col].unique().tolist())

print(f'✓ Loaded: {len(df)} proverbs, {len(all_labels)} categories')


Downloading...
From: https://drive.google.com/uc?id=1Rp9Fg1BitlAR3d1JyH4DmTp3aSKr8H1t
To: /content/swahiliproverbs.csv
100%|██████████| 290k/290k [00:00<00:00, 72.5MB/s]

✓ Loaded: 5060 proverbs, 56 categories


In [ ]:
# Load Llama-3-8B-Instruct
print('Loading Llama-3-8B-Instruct (float16, may take 5-10 minutes)...')
MODEL_NAME = 'meta-llama/Meta-Llama-3-8B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map='auto'
)
print('✓ Llama-3 loaded')


Loading Llama-3-8B-Instruct (float16, may take 5-10 minutes)...


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

✓ Llama-3 loaded


In [ ]:
# Generation and label extraction helpers
def generate(prompt, max_new_tokens=15):
    messages = [
        {'role': 'system',
         'content': 'You are an expert in Swahili language and proverbs. Classify Swahili proverbs into categories. Respond with ONLY the category name, nothing else.'},
        {'role': 'user', 'content': prompt}
    ]
    text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=2048).to(model.device)
    input_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    return response


def extract_label(response, candidates):
    r = response.lower().strip()
    # Exact match
    for label in candidates:
        if label.lower() == r:
            return label
    # Substring match
    for label in candidates:
        if label.lower() in r:
            return label
    # Word-level match
    words = r.split()
    for label in candidates:
        for word in label.lower().split():
            if word in words and len(word) > 3:
                return label
    return None

print('✓ Generation and extraction helpers defined')


✓ Generation and extraction helpers defined


In [ ]:
# Zero-shot classification
print('='*60); print('ZERO-SHOT CLASSIFICATION'); print('='*60)

categories  = sorted(df[label_col].unique().tolist())
cat_string  = '\n'.join([f'- {c}' for c in categories])

sample_zero = df.sample(n=20, random_state=seed)
zero_preds, zero_targets, zero_raw = [], [], []

print(f'Testing {len(sample_zero)} samples | {len(categories)} categories\n')

for idx, (_, row) in enumerate(sample_zero.iterrows()):
    text = row[text_col]; true_label = row[label_col]
    prompt = f'''Classify this Swahili proverb into one of these categories:\n\n{cat_string}\n\nProverb: "{text}"\n\nRespond with ONLY the category name from the list above.'''

    response = generate(prompt)
    pred_label = extract_label(response, categories)

    zero_raw.append({'proverb': text, 'true_label': true_label,
                     'predicted': pred_label, 'correct': pred_label == true_label})

    if pred_label and pred_label in le.classes_:
        zero_preds.append(le.transform([pred_label])[0])
        zero_targets.append(le.transform([true_label])[0])

    pred_str = pred_label if pred_label else 'None'
    print(f'  {idx+1}/20 | Pred: {pred_str:<30} | True: {true_label}')

if zero_preds:
    zero_acc = accuracy_score(zero_targets, zero_preds)
    zero_f1  = f1_score(zero_targets, zero_preds, average='macro', zero_division=0)
    zero_mcc = matthews_corrcoef(zero_targets, zero_preds)
    print(f'\n✓ Zero-shot Acc={zero_acc:.4f} | Macro-F1={zero_f1:.4f} | MCC={zero_mcc:.4f} | Valid: {len(zero_preds)}/20')
    print(f'\nPer-class report (Zero-shot):\n')
    print(classification_report(zero_targets, zero_preds, zero_division=0))
else:
    zero_acc, zero_f1, zero_mcc = 0, 0, 0
    print('No valid predictions')


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


ZERO-SHOT CLASSIFICATION
Testing 20 samples | 56 categories

  1/20 | Pred: Appearance - Beauty            | True: Kindness
  2/20 | Pred: Contentment                    | True: Misfortune
  3/20 | Pred: Death                          | True: Prudence
  4/20 | Pred: Appearance - Beauty            | True: Contentment
  5/20 | Pred: Appearance - Beauty            | True: Foolishness - wisdom
  6/20 | Pred: Appearance - Beauty            | True: Misfortune
  7/20 | Pred: Hunger - food                  | True: Work - laziness
  8/20 | Pred: Appearance - Beauty            | True: Master - servant
  9/20 | Pred: Appearance - Beauty            | True: Prudence
  10/20 | Pred: Cooperation                    | True: Abuse
  11/20 | Pred: Foolishness - wisdom           | True: Excellence - inferiority
  12/20 | Pred: Appearance - Beauty            | True: Gratitude
  13/20 | Pred: Lying - truth                  | True: Lying - truth
  14/20 | Pred: Stealing                       | True: Stealing

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import gdown # Import gdown for downloading
import random # Import random for seed
import numpy as np # Import numpy for seed
import torch # For generate function
from transformers import AutoTokenizer, AutoModelForCausalLM # For generate function
from huggingface_hub import login # For HuggingFace login
from getpass import getpass # For HuggingFace login

print('='*60); print('FEW-SHOT CLASSIFICATION'); print('='*60)

# Re-defining data loading and processing variables (originally from YhYkNyQ8NDTw)
# Ensure the file is downloaded before reading
FILE_ID = '1Rp9Fg1BitlAR3d1JyH4DmTp3aSKr8H1t'
gdown.download(
    f'https://drive.google.com/uc?id={FILE_ID}',
    '/content/swahiliproverbs.csv',
    quiet=True, fuzzy=True # Set quiet=True to suppress download messages
)

df = pd.read_csv('/content/swahiliproverbs.csv')
df = df[['swahili_proverb', 'label']].dropna().copy()
df['swahili_proverb'] = df['swahili_proverb'].astype(str).str.strip()
df['label']           = df['label'].astype(str).str.strip()
df = df[df['swahili_proverb'] != ''].reset_index(drop=True)

text_col, label_col = 'swahili_proverb', 'label'
le = LabelEncoder()
le.fit(df[label_col])
all_labels = sorted(df[label_col].unique().tolist()) # Use all_labels for consistency

categories = all_labels # Assign categories from all_labels
cat_string = '\n'.join([f'- {c}' for c in categories])

# Define seed for reproducibility, as it's used in sampling
seed = 42
np.random.seed(seed); random.seed(seed)

# HuggingFace Authentication for gated models
hf_token = getpass('Paste your HuggingFace token (huggingface.co/settings/tokens): ')
login(token=hf_token)
print('✓ Authenticated with HuggingFace within Few-shot cell')

# Load Llama-3-8B-Instruct (Ensuring tokenizer and model are available)
# Note: This duplicates loading if GSuwpYnYNDTw is also run.
# For robustness against NameError, it's included here.
print('Loading Llama-3-8B-Instruct (float16, may take 5-10 minutes)...')
MODEL_NAME = 'meta-llama/Meta-Llama-3-8B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map='auto'
)
print('✓ Llama-3 loaded within Few-shot cell')

# Generation and label extraction helpers (copied from cell 5ZMlYGymNDTw)
def generate(prompt, max_new_tokens=15):
    messages = [
        {'role': 'system',
         'content': 'You are an expert in Swahili language and proverbs. Classify Swahili proverbs into categories. Respond with ONLY the category name, nothing else.'},
        {'role': 'user', 'content': prompt}
    ]
    text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=2048).to(model.device)
    input_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    return response


def extract_label(response, candidates):
    r = response.lower().strip()
    # Exact match
    for label in candidates:
        if label.lower() == r:
            return label
    # Substring match
    for label in candidates:
        if label.lower() in r:
            return label
    # Word-level match
    words = r.split()
    for label in candidates:
        for word in label.lower().split():
            if word in words and len(word) > 3:
                return label
    return None


few_shot_examples = {}
for label in categories:
    label_df = df[df[label_col] == label]
    few_shot_examples[label] = label_df.sample(
        min(2, len(label_df)), random_state=seed
    )[text_col].tolist()

categories_few = categories[:10]  # Use first 10 categories due to context length
sample_few = df.sample(n=20, random_state=seed+1)
few_preds, few_targets, few_raw = [], [], []

print(f'Testing {len(sample_few)} samples | Examples shown for {len(categories_few)} categories | All {len(categories)} categories available\n')

for idx, (_, row) in enumerate(sample_few.iterrows()):
    text = row[text_col]; true_label = row[label_col]

    examples_text = ''
    for label in categories_few:
        examples_text += f'\nCategory: {label}\n'
        for ex in few_shot_examples[label]:
            examples_text += f'  Example: "{ex}"\n'

    prompt = f'''You are classifying Swahili proverbs. Here are some examples:\n{examples_text}\nAll possible categories:\n{cat_string}\n\nNow classify this proverb:\nProverb: "{text}"\n\nRespond with ONLY the category name from the list above.'''

    response = generate(prompt, max_new_tokens=15)
    pred_label = extract_label(response, categories)

    few_raw.append({'proverb': text, 'true_label': true_label,
                    'predicted': pred_label, 'correct': pred_label == true_label})

    if pred_label and pred_label in le.classes_:
        few_preds.append(le.transform([pred_label])[0])
        few_targets.append(le.transform([true_label])[0])

    pred_str = pred_label if pred_label else 'None'
    print(f'  {idx+1}/20 | Pred: {pred_str:<30} | True: {true_label}')

if few_preds:
    few_acc = accuracy_score(few_targets, few_preds)
    few_f1  = f1_score(few_targets, few_preds, average='macro', zero_division=0)
    few_mcc = matthews_corrcoef(few_targets, few_preds)
    print(f'\n✓ Few-shot Acc={few_acc:.4f} | Macro-F1={few_f1:.4f} | MCC={few_mcc:.4f} | Valid: {len(few_preds)}/20')
    print(f'\nPer-class report (Few-shot):\n')
    print(classification_report(few_targets, few_preds, average='macro', zero_division=0))
else:
    few_acc, few_f1, few_mcc = 0, 0, 0
    print('No valid predictions')

FEW-SHOT CLASSIFICATION
✓ Authenticated with HuggingFace within Few-shot cell
Loading Llama-3-8B-Instruct (float16, may take 5-10 minutes)...


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

...

  14/20 | Pred: Poverty - wealth               | True: Association


[transformers] Both `max_new_tokens` (=15) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  15/20 | Pred: Hurry - patience               | True: Altertness


[transformers] Both `max_new_tokens` (=15) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  16/20 | Pred: Altertness                     | True: Speech - silence


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Results summary and visualization
print('\n' + '='*60)
print('TASK 4 — FINAL RESULTS')
print('='*60)

task4_df = pd.DataFrame([
    {'Method': 'Zero-shot', 'Accuracy': zero_acc, 'Macro-F1': zero_f1, 'MCC': zero_mcc, 'Valid': len(zero_preds)},
    {'Method': 'Few-shot',  'Accuracy': few_acc,  'Macro-F1': few_f1,  'MCC': few_mcc,  'Valid': len(few_preds)},
])
print(task4_df.to_string(index=False))
task4_df.to_csv('/content/task4_results.csv', index=False)

print('\nZero-shot sample predictions:')
print(pd.DataFrame(zero_raw)[['proverb', 'true_label', 'predicted', 'correct']].head(10).to_string(index=False))

print('\nFew-shot sample predictions:')
print(pd.DataFrame(few_raw)[['proverb', 'true_label', 'predicted', 'correct']].head(10).to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].bar(task4_df['Method'], task4_df['Accuracy'], color=['#9b59b6', '#6c3483'], width=0.4)
axes[0].set_title('Accuracy: Zero-shot vs Few-shot', fontweight='bold')
axes[0].set_ylabel('Accuracy'); axes[0].set_ylim(0, max(task4_df['Accuracy'].max() + 0.1, 0.35))
for i, v in enumerate(task4_df['Accuracy']):
    axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')

axes[1].bar(task4_df['Method'], task4_df['Macro-F1'], color=['#9b59b6', '#6c3483'], width=0.4)
axes[1].set_title('Macro-F1: Zero-shot vs Few-shot', fontweight='bold')
axes[1].set_ylabel('Macro-F1'); axes[1].set_ylim(0, max(task4_df['Macro-F1'].max() + 0.1, 0.35))
for i, v in enumerate(task4_df['Macro-F1']):
    axes[1].text(i, v + 0.005, f'{v:.3f}', ha='center', fontweight='bold')

axes[2].bar(task4_df['Method'], task4_df['MCC'], color=['#9b59b6', '#6c3483'], width=0.4)
axes[2].set_title('MCC: Zero-shot vs Few-shot', fontweight='bold')
axes[2].set_ylabel('MCC'); axes[2].set_ylim(min(task4_df['MCC'].min() - 0.05, -0.05), max(task4_df['MCC'].max() + 0.1, 0.35))
for i, v in enumerate(task4_df['MCC']):
    axes[2].text(i, v + 0.005, f'{v:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('/content/task4_results.png', dpi=120, bbox_inches='tight')
plt.show()


TASK 4 — FINAL RESULTS


NameError: name 'pd' is not defined

---
## Experiment Tracking — Logged at the End

The next cell logs all metrics, predictions, and artifacts to W&B and MLflow. The main task code is uncontaminated.




In [ ]:
# Log everything to W&B and MLflow

CONFIG = {
    'task': 'Task 4',
    'model': MODEL_NAME, 'precision': 'float16',
    'sample_size': 20, 'max_new_tokens': 15,
    'do_sample': False, 'temperature': 0.0,
    'num_classes': len(categories),
    'few_shot_categories_shown': 10
}

os.makedirs('/content/mlruns', exist_ok=True)
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
mlflow.set_tracking_uri('file:///content/mlruns')

wandb_key = getpass('Paste W&B API key (wandb.ai/authorize): ')
wandb.login(key=wandb_key)

# ── 1. W&B ────────────────────────────────────────────────────
wandb.init(project='nalapro-swahili', name='task4-llama3-prompting',
           config=CONFIG, reinit=True)

wandb.log({
    'zero_shot/test_acc':    zero_acc,
    'zero_shot/test_f1':     zero_f1,
    'zero_shot/test_mcc':    zero_mcc,
    'zero_shot/valid_preds': len(zero_preds),
    'few_shot/test_acc':     few_acc,
    'few_shot/test_f1':      few_f1,
    'few_shot/test_mcc':     few_mcc,
    'few_shot/valid_preds':  len(few_preds),
    'comparison/zero_minus_few_acc_pct': (zero_acc - few_acc) * 100,
    'comparison/zero_minus_few_f1_pct':  (zero_f1  - few_f1)  * 100,
    'comparison/zero_minus_few_mcc':     (zero_mcc - few_mcc)
})

# Prediction tables
zero_table = wandb.Table(
    columns=['proverb', 'true_label', 'predicted', 'correct'],
    data=[[r['proverb'], r['true_label'], str(r['predicted']), r['correct']] for r in zero_raw]
)
few_table = wandb.Table(
    columns=['proverb', 'true_label', 'predicted', 'correct'],
    data=[[r['proverb'], r['true_label'], str(r['predicted']), r['correct']] for r in few_raw]
)
wandb.log({'zero_shot/predictions': zero_table})
wandb.log({'few_shot/predictions':  few_table})

wandb.log({'task4_summary': wandb.Table(dataframe=task4_df[['Method','Accuracy','Macro-F1','MCC','Valid']])})
wandb.log({'task4_chart':   wandb.Image('/content/task4_results.png')})

wandb_url = wandb.run.url
wandb.finish()

# ── 3. MLflow ────────────────────────────────────────────────
mlflow.set_experiment('nalapro-task4-llama')
for method, acc, f1, mcc, valid, raw in [
    ('zero-shot', zero_acc, zero_f1, zero_mcc, len(zero_preds), zero_raw),
    ('few-shot',  few_acc,  few_f1,  few_mcc,  len(few_preds),  few_raw)
]:
    with mlflow.start_run(run_name=f'task4-{method}'):
        mlflow.log_params({**CONFIG, 'method': method})
        mlflow.log_metric('test_acc',    acc)
        mlflow.log_metric('test_f1',     f1)
        mlflow.log_metric('test_mcc',    mcc)
        mlflow.log_metric('valid_preds', valid)
        # Save predictions as CSV artifact
        pd.DataFrame(raw).to_csv(f'/content/task4_{method}_predictions.csv', index=False)
        mlflow.log_artifact(f'/content/task4_{method}_predictions.csv')

with mlflow.start_run(run_name='task4-summary'):
    mlflow.log_metric('zero_minus_few_acc_pct', (zero_acc - few_acc) * 100)
    mlflow.log_metric('zero_minus_few_f1_pct',  (zero_f1  - few_f1)  * 100)
    mlflow.log_artifact('/content/task4_results.csv')
    mlflow.log_artifact('/content/task4_results.png')

print('\n' + '='*60)
print('✓ ALL TRACKING COMPLETE')
print('='*60)
print(f'W&B:    {wandb_url}')
print('Aim:    Run  ! up --repo /content/. ')
print('MLflow: Run  !mlflow ui --backend-store-uri /content/mlruns')
